# Studi Komparatif Model Generatif (Vanilla AE, Sparse AE, VAE, GAN, DCGAN, DDPM) + Batasan MLP Classifier
Deep Learning (D6421) Sesi 07 | Dataset: Fashion-MNIST | Seed 42

**Cara pakai:** Runtime > Change runtime type > **T4 GPU**, lalu *Run all* (atau jalankan berurutan dari atas).
- Sel training otomatis **melewati** model yang checkpoint-nya sudah ada di `checkpoints/`, jadi aman dijalankan ulang.
- Checkpoint, hasil, dan figur disimpan ke Google Drive (folder `dl_sesi07`) agar tidak hilang saat sesi Colab reset.
- Jika Anda punya cadangan `checkpoints/` lama, taruh di `MyDrive/dl_sesi07/checkpoints/` **sebelum** menjalankan sel training.

## 0. Setup

In [ ]:
# Pasang library SEBELUM ada import torchmetrics apa pun (jika sudah terlanjur, restart session lalu jalankan ulang).
!pip install -q torchmetrics torch-fidelity

import os, torch
assert torch.cuda.is_available(), "GPU TIDAK AKTIF! Runtime > Change runtime type > T4 GPU, lalu jalankan ulang dari atas."
device = "cuda"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))
os.makedirs("models", exist_ok=True)

In [ ]:
# Simpan checkpoints/results/figures di Google Drive agar tahan reset sesi.
USE_DRIVE = True
import os, shutil
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/dl_sesi07"
    for d in ["checkpoints", "results", "figures"]:
        os.makedirs(f"{BASE}/{d}", exist_ok=True)
        if os.path.isdir(d) and not os.path.islink(d):          # pindahkan isi folder lokal lama ke Drive
            shutil.copytree(d, f"{BASE}/{d}", dirs_exist_ok=True)
            shutil.rmtree(d)
        if not os.path.exists(d):
            os.symlink(f"{BASE}/{d}", d)
else:
    for d in ["checkpoints", "results", "figures"]:
        os.makedirs(d, exist_ok=True)
print(os.listdir("checkpoints"))

## 1. Berkas kode modular (ditulis ke disk; semuanya ikut di-commit ke GitHub)

In [ ]:
%%writefile data.py
import torch, numpy as np, random
from torch.utils.data import DataLoader, ConcatDataset, random_split
from torchvision import datasets, transforms

SEED = 42

def set_seed(seed=SEED):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

def get_loaders(batch_size=128, root="./data"):
    set_seed()
    tf = transforms.ToTensor()  # piksel jadi rentang [0, 1]
    train_raw = datasets.FashionMNIST(root, train=True,  download=True, transform=tf)
    test_raw  = datasets.FashionMNIST(root, train=False, download=True, transform=tf)

    full = ConcatDataset([train_raw, test_raw])      # 70.000 citra
    n = len(full)
    n_train, n_val = int(0.8 * n), int(0.1 * n)
    n_test = n - n_train - n_val

    train_set, val_set, test_set = random_split(
        full, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(SEED)
    )

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [ ]:
%%writefile models/ae.py
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        # Encoder: 784 -> 256 -> 64
        self.encoder = nn.Sequential(
            nn.Flatten(),                 # [B,1,28,28] -> [B,784]
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, latent_dim),   # latent tanpa aktivasi
        )
        # Decoder: 64 -> 256 -> 784
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid(),                 # output di rentang [0, 1]
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

In [ ]:
%%writefile models/sparse_ae.py
import torch.nn as nn

class SparseAutoencoder(nn.Module):
    def __init__(self, latent_dim=64, hidden_dim=256):
        super().__init__()
        self.flatten = nn.Flatten()
        self.enc1 = nn.Sequential(nn.Linear(784, hidden_dim), nn.ReLU())  # h: lapisan sparse
        self.enc2 = nn.Linear(hidden_dim, latent_dim)                      # z: bottleneck
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 784),
            nn.Sigmoid(),
        )

    def encode_with_h(self, x):
        h = self.enc1(self.flatten(x))
        return self.enc2(h), h

    def encode(self, x):
        return self.encode_with_h(x)[0]

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):                 # kompatibel dengan evaluate_recon
        z = self.encode(x)
        return self.decode(z), z

    def forward_with_h(self, x):          # dipakai saat training (butuh h untuk penalti L1)
        z, h = self.encode_with_h(x)
        return self.decode(z), z, h

In [ ]:
%%writefile models/vae.py
import torch, torch.nn as nn

class VAE(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.enc = nn.Sequential(nn.Flatten(), nn.Linear(784, 256), nn.ReLU())
        self.fc_mu = nn.Linear(256, latent_dim)
        self.fc_logvar = nn.Linear(256, latent_dim)
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def encode(self, x):
        h = self.enc(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z):
        return self.dec(z).view(-1, 1, 28, 28)

    def forward(self, x):                      # kompatibel dengan evaluate_recon
        mu, _ = self.encode(x)                 # rekonstruksi deterministik memakai mean
        return self.decode(mu), mu

    def forward_train(self, x):                # dipakai saat training
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [ ]:
%%writefile models/mlp.py
import torch.nn as nn

class MLPClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 10)
        self.relu = nn.ReLU()

    def features(self, x):                 # z_MLP (64 dim), sebelum lapisan klasifikasi akhir
        h1 = self.relu(self.fc1(self.flatten(x)))
        return self.relu(self.fc2(h1))

    def forward(self, x):
        return self.fc3(self.features(x))

In [ ]:
%%writefile models/frozen_decoder.py
import torch, torch.nn as nn

class FrozenLatentDecoder(nn.Module):
    def __init__(self, clf):
        super().__init__()
        self.clf = clf
        for p in self.clf.parameters():
            p.requires_grad = False                 # bekukan seluruh bobot classifier
        self.decoder = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid(),
        )

    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)

    def forward(self, x):                           # kompatibel dengan evaluate_recon
        with torch.no_grad():
            z = self.clf.features(x)                # z_MLP, tidak ikut dilatih
        return self.decode(z), z

In [ ]:
%%writefile models/gan.py
import torch.nn as nn

class VanillaG(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 784), nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

class VanillaD(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, 512), nn.LeakyReLU(0.2),
            nn.Linear(512, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 1),            # logit, tanpa sigmoid
        )
    def forward(self, x):
        return self.net(x)

In [ ]:
%%writefile models/dcgan.py
import torch.nn as nn

class DCGenerator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.latent_dim = latent_dim
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 128, 7, 1, 0, bias=False),  # 1x1 -> 7x7
            nn.BatchNorm2d(128), nn.ReLU(True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),          # 7 -> 14
            nn.BatchNorm2d(64), nn.ReLU(True),
            nn.ConvTranspose2d(64, 1, 4, 2, 1),                        # 14 -> 28
            nn.Tanh(),
        )
    def forward(self, z):
        return self.net(z.view(z.size(0), -1, 1, 1))

class DCDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2, True),                                      # 28 -> 14
            nn.Conv2d(64, 128, 4, 2, 1, bias=False), nn.BatchNorm2d(128), nn.LeakyReLU(0.2, True),   # 14 -> 7
            nn.Conv2d(128, 1, 7, 1, 0),                                                              # 7 -> 1
        )
    def forward(self, x):
        return self.net(x).view(-1, 1)

def init_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

In [ ]:
%%writefile models/ddpm.py
import math, torch, torch.nn as nn
import torch.nn.functional as F

def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.t_proj = nn.Linear(t_dim, out_ch)
        self.norm2 = nn.GroupNorm(8, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.t_proj(t_emb)[:, :, None, None]     # suntikkan informasi langkah t
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)

class UNet(nn.Module):
    def __init__(self, base=64, t_dim=128):
        super().__init__()
        self.t_dim = t_dim
        self.t_mlp = nn.Sequential(nn.Linear(t_dim, t_dim), nn.SiLU(), nn.Linear(t_dim, t_dim))
        self.inc = nn.Conv2d(1, base, 3, padding=1)
        self.down1 = ResBlock(base, base, t_dim)            # 28x28
        self.down2 = ResBlock(base, base * 2, t_dim)        # 14x14
        self.mid = ResBlock(base * 2, base * 2, t_dim)      # 7x7
        self.up2 = ResBlock(base * 4, base, t_dim)          # 14x14 (hasil concat)
        self.up1 = ResBlock(base * 2, base, t_dim)          # 28x28 (hasil concat)
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, 1, 3, padding=1))

    def forward(self, x, t):
        te = self.t_mlp(timestep_embedding(t, self.t_dim))
        d1 = self.down1(self.inc(x), te)                     # [B, 64, 28, 28]
        d2 = self.down2(F.avg_pool2d(d1, 2), te)             # [B, 128, 14, 14]
        m = self.mid(F.avg_pool2d(d2, 2), te)                # [B, 128, 7, 7]
        u2 = self.up2(torch.cat([F.interpolate(m, scale_factor=2), d2], 1), te)    # [B, 64, 14, 14]
        u1 = self.up1(torch.cat([F.interpolate(u2, scale_factor=2), d1], 1), te)   # [B, 64, 28, 28]
        return self.out(u1)                                  # prediksi noise, bentuk sama dengan x

In [ ]:
%%writefile train_utils.py
import time, torch
import torch.nn.functional as F

@torch.no_grad()
def val_mse(model, loader, device):
    model.eval()
    tot = 0.0
    for x, _ in loader:
        x = x.to(device)
        x_hat, _ = model(x)
        tot += F.mse_loss(x_hat, x).item() * x.size(0)
    return tot / len(loader.dataset)

def train_ae(model, train_loader, val_loader, device, epochs=20, lr=1e-3):
    """Vanilla AE: loss = MSE(x, x_hat)."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train": [], "val": [], "epoch_time": []}
    for ep in range(epochs):
        t0 = time.time(); model.train(); tot = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            x_hat, _ = model(x)
            loss = F.mse_loss(x_hat, x)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0)
        tr, va, dt = tot / len(train_loader.dataset), val_mse(model, val_loader, device), time.time() - t0
        hist["train"].append(tr); hist["val"].append(va); hist["epoch_time"].append(dt)
        print(f"Epoch {ep+1:2d}/{epochs} | train {tr:.5f} | val {va:.5f} | {dt:.1f}s")
    return hist

def train_sparse_ae(model, train_loader, val_loader, device, epochs=20, lr=1e-3, lam=1e-4):
    """Sparse AE: loss = MSE + lambda * ||h||_1 (h = lapisan tersembunyi 256 unit)."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train": [], "val": [], "train_l1": [], "epoch_time": []}
    N = len(train_loader.dataset)
    for ep in range(epochs):
        t0 = time.time(); model.train(); tot = tot_l1 = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            x_hat, z, h = model.forward_with_h(x)
            mse = F.mse_loss(x_hat, x)
            l1 = h.abs().sum(dim=1).mean()          # ||h||_1 per gambar, dirata-rata per batch
            loss = mse + lam * l1
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0); tot_l1 += l1.item() * x.size(0)
        va, dt = val_mse(model, val_loader, device), time.time() - t0
        hist["train"].append(tot / N); hist["val"].append(va)
        hist["train_l1"].append(tot_l1 / N); hist["epoch_time"].append(dt)
        print(f"Epoch {ep+1:2d}/{epochs} | train(total) {tot/N:.5f} | L1 {tot_l1/N:.2f} | val MSE {va:.5f} | {dt:.1f}s")
    return hist

def vae_loss(x_hat, x, mu, logvar):
    recon = F.binary_cross_entropy(x_hat, x, reduction="sum") / x.size(0)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / x.size(0)
    return recon + kl, recon, kl

def train_vae(model, train_loader, val_loader, device, epochs=20, lr=1e-3):
    """VAE: loss = BCE rekonstruksi + KL(N(mu, sigma^2) || N(0, I))."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"train": [], "train_recon": [], "train_kl": [], "val_mse": [], "epoch_time": []}
    N = len(train_loader.dataset)
    for ep in range(epochs):
        t0 = time.time(); model.train(); tot = tot_r = tot_k = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            x_hat, mu, logvar = model.forward_train(x)
            loss, recon, kl = vae_loss(x_hat, x, mu, logvar)
            opt.zero_grad(); loss.backward(); opt.step()
            n = x.size(0)
            tot += loss.item() * n; tot_r += recon.item() * n; tot_k += kl.item() * n
        va, dt = val_mse(model, val_loader, device), time.time() - t0
        hist["train"].append(tot / N); hist["train_recon"].append(tot_r / N)
        hist["train_kl"].append(tot_k / N); hist["val_mse"].append(va); hist["epoch_time"].append(dt)
        print(f"Epoch {ep+1:2d}/{epochs} | loss {tot/N:.2f} | recon {tot_r/N:.2f} | KL {tot_k/N:.2f} | val MSE {va:.5f} | {dt:.1f}s")
    return hist

@torch.no_grad()
def accuracy(clf, loader, device):
    clf.eval(); correct = 0
    for x, y in loader:
        correct += (clf(x.to(device)).argmax(1) == y.to(device)).sum().item()
    return correct / len(loader.dataset)

def train_mlp(clf, train_loader, val_loader, device, epochs=20, lr=1e-3):
    """MLP Classifier 784-256-64-10 dengan Cross-Entropy."""
    opt = torch.optim.Adam(clf.parameters(), lr=lr)
    hist = {"train_loss": [], "val_acc": [], "epoch_time": []}
    N = len(train_loader.dataset)
    for ep in range(epochs):
        t0 = time.time(); clf.train(); tot = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            loss = F.cross_entropy(clf(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0)
        acc, dt = accuracy(clf, val_loader, device), time.time() - t0
        hist["train_loss"].append(tot / N); hist["val_acc"].append(acc); hist["epoch_time"].append(dt)
        print(f"Epoch {ep+1:2d}/{epochs} | train loss {tot/N:.4f} | val acc {acc:.4f}")
    return hist

def train_frozen_decoder(fd, train_loader, val_loader, device, epochs=20, lr=1e-3):
    """Hanya decoder yang dilatih; classifier di dalam fd dibekukan."""
    fd.clf.eval()
    opt = torch.optim.Adam(fd.decoder.parameters(), lr=lr)
    hist = {"train": [], "val": [], "epoch_time": []}
    for ep in range(epochs):
        t0 = time.time(); fd.decoder.train(); tot = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            x_hat, _ = fd(x)
            loss = F.mse_loss(x_hat, x)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0)
        fd.decoder.eval()
        va, dt = val_mse(fd, val_loader, device), time.time() - t0
        hist["train"].append(tot / len(train_loader.dataset)); hist["val"].append(va); hist["epoch_time"].append(dt)
        print(f"Epoch {ep+1:2d}/{epochs} | train {hist['train'][-1]:.5f} | val {va:.5f}")
    return hist

In [ ]:
%%writefile gan_utils.py
import time, torch
import torch.nn as nn

def train_gan(G, D, loader, device, epochs=20, latent_dim=100, lr=2e-4):
    bce = nn.BCEWithLogitsLoss()
    optG = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    optD = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    hist = {"loss_D": [], "loss_G": [], "epoch_time": []}
    N = len(loader.dataset)
    for epoch in range(epochs):
        t0 = time.time()
        G.train(); D.train()
        sd = sg = 0.0
        for x, _ in loader:
            x = x.to(device) * 2 - 1                  # [0,1] -> [-1,1]
            b = x.size(0)
            real = torch.ones(b, 1, device=device)
            fake = torch.zeros(b, 1, device=device)

            # --- update D: kenali asli sebagai 1, palsu sebagai 0 ---
            x_fake = G(torch.randn(b, latent_dim, device=device))
            loss_D = bce(D(x), real) + bce(D(x_fake.detach()), fake)
            optD.zero_grad(); loss_D.backward(); optD.step()

            # --- update G (non-saturating): ingin D menilai palsu sebagai 1 ---
            loss_G = bce(D(x_fake), real)
            optG.zero_grad(); loss_G.backward(); optG.step()

            sd += loss_D.item() * b; sg += loss_G.item() * b
        dt = time.time() - t0
        hist["loss_D"].append(sd / N); hist["loss_G"].append(sg / N); hist["epoch_time"].append(dt)
        print(f"Epoch {epoch+1:2d}/{epochs} | loss_D {sd/N:.4f} | loss_G {sg/N:.4f} | {dt:.1f}s")
    return hist

In [ ]:
%%writefile ddpm_utils.py
import time, torch
import torch.nn.functional as F

class Diffusion:
    def __init__(self, T=1000, beta_start=1e-4, beta_end=0.02, device="cpu"):
        self.T = T
        self.betas = torch.linspace(beta_start, beta_end, T, device=device)
        self.alphas = 1 - self.betas
        self.alpha_bar = torch.cumprod(self.alphas, dim=0)

    def q_sample(self, x0, t, noise):
        """Forward process: lompat langsung dari x0 ke x_t."""
        ab = self.alpha_bar[t][:, None, None, None]
        return ab.sqrt() * x0 + (1 - ab).sqrt() * noise

    @torch.no_grad()
    def sample(self, model, n, device, return_traj=False, traj_steps=()):
        """Reverse process: dari noise murni x_T ke gambar x_0."""
        x = torch.randn(n, 1, 28, 28, device=device)
        traj = [x.clone().cpu()]
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=device, dtype=torch.long)
            eps = model(x, t)
            alpha, ab, beta = self.alphas[i], self.alpha_bar[i], self.betas[i]
            mean = (x - beta / (1 - ab).sqrt() * eps) / alpha.sqrt()
            x = mean + beta.sqrt() * torch.randn_like(x) if i > 0 else mean
            if return_traj and i in traj_steps:
                traj.append(x.clone().cpu())
        return (x, traj) if return_traj else x

def train_ddpm(model, ema_model, diffusion, loader, device, epochs=20, lr=2e-4, ema_decay=0.999):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = {"loss": [], "epoch_time": []}
    N = len(loader.dataset)
    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        tot = 0.0
        for x, _ in loader:
            x = x.to(device) * 2 - 1                                   # [0,1] -> [-1,1]
            t = torch.randint(0, diffusion.T, (x.size(0),), device=device)
            noise = torch.randn_like(x)
            x_t = diffusion.q_sample(x, t, noise)
            loss = F.mse_loss(model(x_t, t), noise)                    # prediksi noise
            opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():                                      # update EMA
                for pe, p in zip(ema_model.parameters(), model.parameters()):
                    pe.mul_(ema_decay).add_(p.detach(), alpha=1 - ema_decay)
            tot += loss.item() * x.size(0)
        dt = time.time() - t0
        hist["loss"].append(tot / N); hist["epoch_time"].append(dt)
        print(f"Epoch {epoch+1:2d}/{epochs} | loss {tot/N:.5f} | {dt:.1f}s")
    return hist

In [ ]:
%%writefile evaluate.py
import math
import torch
import torch.nn.functional as F
from torchmetrics.image import StructuralSimilarityIndexMeasure

@torch.no_grad()
def evaluate_recon(model, loader, device):
    model.eval()
    ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
    total_sq_err, n_pixels = 0.0, 0
    for x, _ in loader:
        x = x.to(device)
        x_hat, _ = model(x)
        total_sq_err += F.mse_loss(x_hat, x, reduction="sum").item()
        n_pixels += x.numel()
        ssim_metric.update(x_hat, x)
    mse = total_sq_err / n_pixels
    psnr = 10 * math.log10(1.0 / mse)
    ssim = ssim_metric.compute().item()
    return mse, psnr, ssim

In [ ]:
%%writefile benchmark.py
import time, torch

@torch.no_grad()
def measure_latency(fn, device, warmup=10, repeats=50):
    """Rata-rata waktu (ms) untuk satu pemanggilan fn()."""
    for _ in range(warmup):
        fn()
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeats):
        fn()
    if device == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t0) / repeats * 1000

@torch.no_grad()
def generate_in_batches(sample_fn, n, batch=250):
    out, done = [], 0
    while done < n:
        b = min(batch, n - done)
        out.append(sample_fn(b).clamp(0, 1).cpu())     # kembalikan di rentang [0,1]
        done += b
    return torch.cat(out)

@torch.no_grad()
def compute_fid(fid_metric, fake_images, device, batch=250):
    fid_metric.reset()      # statistik citra asli tetap tersimpan (reset_real_features=False)
    for i in range(0, len(fake_images), batch):
        x = fake_images[i:i + batch].to(device).repeat(1, 3, 1, 1)   # Inception butuh 3 kanal
        fid_metric.update(x, real=False)
    return fid_metric.compute().item()

In [ ]:
%%writefile figures.py
import os, json, torch
import matplotlib.pyplot as plt

os.makedirs("figures", exist_ok=True)

def _to01(a):
    return ((a.clamp(-1, 1) + 1) / 2).cpu()

def fig1_reconstruction(models, x, path="figures/fig1_reconstruction.png"):
    rows = [("Asli", x.cpu())]
    with torch.no_grad():
        for name, m in models.items():
            rows.append((name, m(x)[0].cpu()))
    n = x.size(0)
    fig, axes = plt.subplots(len(rows), n, figsize=(1.5 * n, 1.6 * len(rows)))
    for r, (name, imgs) in enumerate(rows):
        for c in range(n):
            axes[r, c].imshow(imgs[c, 0], cmap="gray", vmin=0, vmax=1)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        axes[r, 0].set_ylabel(name, rotation=0, ha="right", va="center", fontsize=10)
    fig.suptitle("Figur 1. Rekonstruksi citra test set")
    plt.tight_layout(); plt.savefig(path, dpi=200, bbox_inches="tight"); plt.show()

def fig2_samples(samples_dict, path="figures/fig2_samples.png"):
    fig, axes = plt.subplots(1, len(samples_dict), figsize=(4 * len(samples_dict), 4.4))
    for ax, (name, imgs) in zip(axes, samples_dict.items()):
        grid = imgs[:16, 0].reshape(4, 4, 28, 28).permute(0, 2, 1, 3).reshape(112, 112)
        ax.imshow(grid, cmap="gray", vmin=0, vmax=1); ax.axis("off"); ax.set_title(name)
    fig.suptitle("Figur 2. Sampling generatif acak (4x4 per model)")
    plt.tight_layout(); plt.savefig(path, dpi=200, bbox_inches="tight"); plt.show()

def fig3_mlp_limits(clf, fd, ae, test_loader, device, path="figures/fig3_mlp_limits.png"):
    W1, W2, W3 = clf.fc1.weight.data, clf.fc2.weight.data, clf.fc3.weight.data
    x = next(iter(test_loader))[0][:10].to(device)
    names = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]
    with torch.no_grad():
        # Tahap 2: one-hot kelas -> piksel lewat transpos bobot
        y = torch.eye(10, device=device)
        s2 = torch.sigmoid(torch.relu(torch.relu(y @ W3) @ W2) @ W1).view(-1, 1, 28, 28)
        # Tahap 3: z_MLP citra uji -> piksel lewat transpos bobot
        z_mlp = clf.features(x)
        s3 = torch.sigmoid(torch.relu(z_mlp @ W2) @ W1).view(-1, 1, 28, 28)
        # Tahap 4: frozen decoder vs Vanilla AE
        rec_fd, _ = fd(x)
        rec_ae, _ = ae(x)
        # Tahap 5: prior sampling N(mu_z, diag(sd_z^2))
        zs = torch.cat([clf.features(b.to(device)) for b, _ in test_loader])
        mu, sd = zs.mean(0), zs.std(0)
        torch.manual_seed(42)
        samp = fd.decode(mu + sd * torch.randn(10, 64, device=device))
    rows = [("(1) Asli", x), ("(2a) Tahap 2:\none-hot y_c lewat W^T", s2),
            ("(2b) Tahap 3:\nz_MLP lewat W^T", s3), ("(3a) Frozen decoder\n(dari z_MLP)", rec_fd),
            ("(3b) Vanilla AE\n(dari z_AE)", rec_ae), ("(4) Prior sampling\nN(mu_z, diag(sd_z^2))", samp)]
    fig, axes = plt.subplots(len(rows), 10, figsize=(14, 9.5))
    for r, (name, imgs) in enumerate(rows):
        for c in range(10):
            axes[r, c].imshow(imgs[c, 0].cpu(), cmap="gray", vmin=0, vmax=1)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
        axes[r, 0].set_ylabel(name, rotation=0, ha="right", va="center", fontsize=9)
    for c in range(10):
        axes[1, c].set_title(names[c], fontsize=8)     # judul kolom hanya berlaku untuk baris (2a)
    fig.suptitle("Figur 3. Batasan bobot dan fitur laten MLP Classifier untuk image generation")
    plt.tight_layout(); plt.savefig(path, dpi=200, bbox_inches="tight"); plt.show()

def fig4_interpolation(ae, sparse_ae, vae, gan_G, dc_G, clf, test_loader, device,
                       cls_a=1, cls_b=7, path="figures/fig4_interpolation.png"):
    alphas = torch.tensor([0.0, 0.2, 0.4, 0.6, 0.8, 1.0], device=device).view(-1, 1)
    lerp = lambda za, zb: (1 - alphas) * za + alphas * zb
    xs, ys = [], []
    for b, y in test_loader:
        xs.append(b); ys.append(y)
        if len(xs) >= 5: break
    xs, ys = torch.cat(xs), torch.cat(ys)
    xa, xb = xs[ys == cls_a][:1].to(device), xs[ys == cls_b][:1].to(device)

    def find_z(G, cls, dim, n=500):          # GAN tanpa encoder: cari noise yang menghasilkan kelas tertentu
        z = torch.randn(n, dim, device=device)
        pred = clf((G(z) + 1) / 2).argmax(1)
        idx = (pred == cls).nonzero()
        return z[idx[0]] if len(idx) else z[:1]

    rows = []
    with torch.no_grad():
        for name, enc, dec in [("Vanilla AE", ae.encode, ae.decode),
                               ("Sparse AE", sparse_ae.encode, sparse_ae.decode),
                               ("VAE", lambda t: vae.encode(t)[0], vae.decode)]:
            rows.append((name, dec(lerp(enc(xa), enc(xb)))))
        torch.manual_seed(42)
        for name, G in [("Vanilla GAN", gan_G), ("DCGAN", dc_G)]:
            za, zb = find_z(G, cls_a, 100), find_z(G, cls_b, 100)
            rows.append((name, (G(lerp(za, zb)) + 1) / 2))
    fig, axes = plt.subplots(len(rows), 6, figsize=(9, 8))
    for r, (name, imgs) in enumerate(rows):
        for c in range(6):
            axes[r, c].imshow(imgs[c, 0].cpu(), cmap="gray", vmin=0, vmax=1)
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            if r == 0: axes[r, c].set_title(f"alpha={alphas[c].item():.1f}", fontsize=9)
        axes[r, 0].set_ylabel(name, rotation=0, ha="right", va="center", fontsize=10)
    fig.suptitle("Figur 4. Interpolasi linier z(alpha) = (1-alpha) zA + alpha zB")
    plt.tight_layout(); plt.savefig(path, dpi=200, bbox_inches="tight"); plt.show()

def fig5_curves_and_trajectory(diffusion, ema, device, path="figures/fig5_curves_ddpm.png"):
    H = lambda n: json.load(open(f"checkpoints/{n}_history.json"))
    ae, sp, va, gn, dc, dd = [H(n) for n in ["ae", "sparse_ae", "vae", "gan", "dcgan", "ddpm"]]
    ep = lambda h: range(1, len(h) + 1)
    fig = plt.figure(figsize=(15, 13))
    top, bot = fig.subfigures(2, 1, height_ratios=[1, 1.15])
    ax = top.subplots(2, 3).flatten()
    ax[0].plot(ep(ae["train"]), ae["train"], label="train MSE"); ax[0].plot(ep(ae["val"]), ae["val"], label="val MSE")
    ax[0].set_title("Vanilla AE")
    ax[1].plot(ep(sp["train"]), sp["train"], label="train (MSE + lambda*L1)"); ax[1].plot(ep(sp["val"]), sp["val"], label="val MSE")
    ax[1].set_title("Sparse AE")
    ax[2].plot(ep(va["train"]), va["train"], label="total"); ax[2].plot(ep(va["train_recon"]), va["train_recon"], label="rekonstruksi (BCE)")
    ax[2].plot(ep(va["train_kl"]), va["train_kl"], label="KL"); ax[2].set_title("VAE (loss per gambar)")
    for a, h, t in [(ax[3], gn, "Vanilla GAN"), (ax[4], dc, "DCGAN")]:
        a.plot(ep(h["loss_D"]), h["loss_D"], label="loss D"); a.plot(ep(h["loss_G"]), h["loss_G"], label="loss G")
        a.axhline(1.386, ls="--", c="gray", lw=0.8); a.axhline(0.693, ls=":", c="gray", lw=0.8); a.set_title(t)
    ax[5].plot(ep(dd["loss"]), dd["loss"], label="MSE prediksi noise"); ax[5].set_title("DDPM")
    for a in ax:
        a.set_xlabel("epoch"); a.legend(fontsize=8); a.grid(alpha=0.3)
    top.suptitle("Figur 5A. Kurva training loss keenam model (garis putus-putus GAN: 2 ln2 dan ln2)")

    steps = [800, 600, 400, 200, 100, 50, 0]
    torch.manual_seed(42)
    _, traj = diffusion.sample(ema, 4, device, return_traj=True, traj_steps=set(steps))
    labels = ["x_T"] + [f"t={s}" for s in steps]
    axs = bot.subplots(4, len(labels))
    for r in range(4):
        for c, snap in enumerate(traj):
            axs[r, c].imshow(_to01(snap[r, 0]), cmap="gray"); axs[r, c].axis("off")
            if r == 0: axs[r, c].set_title(labels[c], fontsize=9)
    bot.suptitle("Figur 5B. Lintasan reverse diffusion DDPM (x_T menuju x_0)")
    fig.savefig(path, dpi=200, bbox_inches="tight"); plt.show()

In [ ]:
%%writefile requirements.txt
torch
torchvision
numpy
matplotlib
pandas
torchmetrics
torch-fidelity

In [ ]:
%%writefile README.md
# Studi Komparatif Model Generatif: AE, Sparse AE, VAE, GAN, DCGAN, DDPM

Tugas Take-Home Deep Learning (D6421) Sesi 07. Dataset: Fashion-MNIST (split 80/10/10, seed 42).

## Struktur
- `data.py` : dataset, split 80/10/10, seed tetap
- `models/` : ae, sparse_ae, vae, mlp, frozen_decoder, gan, dcgan, ddpm
- `train_utils.py`, `gan_utils.py`, `ddpm_utils.py` : fungsi training
- `evaluate.py` : MSE / PSNR / SSIM; `benchmark.py` : latensi dan FID
- `figures.py` : Figur 1-5
- `Deeplearning.ipynb` : notebook yang menjalankan seluruh pipeline

## Cara menjalankan
1. Buka `Deeplearning.ipynb` di Google Colab, pilih Runtime > T4 GPU.
2. `pip install -r requirements.txt` (sudah ada di sel pertama notebook).
3. Jalankan seluruh sel dari atas ke bawah. Sel training otomatis melewati model yang
   checkpoint-nya sudah ada di `checkpoints/`.
4. Hasil tabel ada di `results/`, figur di `figures/`.

Estimasi waktu di GPU T4: AE/VAE ~2.5 menit, GAN ~3 menit, DCGAN ~4 menit, DDPM ~12 menit
(training), FID ~20 menit (DDPM paling lama).

## 2. Data: split 80/10/10, seed tetap

In [ ]:
from data import get_loaders, set_seed
import matplotlib.pyplot as plt

train_loader, val_loader, test_loader = get_loaders()
print("Train:", len(train_loader.dataset), "| Val:", len(val_loader.dataset), "| Test:", len(test_loader.dataset))  # 56000 / 7000 / 7000
x, y = next(iter(train_loader))
print(x.shape, x.min().item(), x.max().item())
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.imshow(x[i, 0], cmap="gray"); ax.set_title(int(y[i])); ax.axis("off")
plt.show()

## 3. Training (otomatis skip jika checkpoint sudah ada)

In [ ]:
import json, os, torch
from data import get_loaders, set_seed
from train_utils import train_ae, train_sparse_ae, train_vae, train_mlp, train_frozen_decoder, accuracy
from models.ae import Autoencoder
from models.sparse_ae import SparseAutoencoder
from models.vae import VAE
from models.mlp import MLPClassifier
from models.frozen_decoder import FrozenLatentDecoder

EPOCHS = 20          # sama untuk SEMUA model agar perbandingan adil
LAMBDA = 1e-4        # kekuatan penalti sparsity

def run_or_load(name, make_model, trainer):
    """Latih model jika checkpoint belum ada; jika ada, langsung dimuat."""
    ck = f"checkpoints/{name}.pt"
    set_seed()
    model = make_model().to(device)
    if os.path.exists(ck):
        model.load_state_dict(torch.load(ck, map_location=device))
        print(f"[skip] {name}: memuat checkpoint yang sudah ada")
    else:
        print(f"[train] {name}")
        hist = trainer(model)
        torch.save(model.state_dict(), ck)
        json.dump(hist, open(f"checkpoints/{name}_history.json", "w"))
    return model.eval()

ae        = run_or_load("ae",        lambda: Autoencoder(64),       lambda m: train_ae(m, train_loader, val_loader, device, EPOCHS))
sparse_ae = run_or_load("sparse_ae", lambda: SparseAutoencoder(64), lambda m: train_sparse_ae(m, train_loader, val_loader, device, EPOCHS, lam=LAMBDA))
vae       = run_or_load("vae",       lambda: VAE(64),               lambda m: train_vae(m, train_loader, val_loader, device, EPOCHS))
clf       = run_or_load("mlp",       lambda: MLPClassifier(),       lambda m: train_mlp(m, train_loader, val_loader, device, EPOCHS))
print(f"\nMLP Classifier TEST ACCURACY: {accuracy(clf, test_loader, device):.4f}  (syarat >= 0.8800)")

# Frozen latent decoder (Tahap 4): classifier dibekukan, hanya decoder yang dilatih
fd = run_or_load("frozen_decoder", lambda: FrozenLatentDecoder(clf), lambda m: train_frozen_decoder(m, train_loader, val_loader, device, EPOCHS))

In [ ]:
from gan_utils import train_gan
from models.gan import VanillaG, VanillaD
from models.dcgan import DCGenerator, DCDiscriminator, init_weights

def run_or_load_gan(name, G_cls, D_cls, init=None):
    gk, dk, hk = f"checkpoints/{name}_G.pt", f"checkpoints/{name}_D.pt", f"checkpoints/{name}_history.json"
    set_seed()
    G, D = G_cls(100).to(device), D_cls().to(device)
    if init: G.apply(init); D.apply(init)
    if os.path.exists(gk):
        G.load_state_dict(torch.load(gk, map_location=device))
        print(f"[skip] {name}: memuat checkpoint yang sudah ada")
    else:
        print(f"[train] {name}")
        hist = train_gan(G, D, train_loader, device, epochs=EPOCHS, latent_dim=100)
        torch.save(G.state_dict(), gk); torch.save(D.state_dict(), dk)
        json.dump(hist, open(hk, "w"))
    return G.eval()

gan_G = run_or_load_gan("gan",   VanillaG,    VanillaD)
dc_G  = run_or_load_gan("dcgan", DCGenerator, DCDiscriminator, init=init_weights)

In [ ]:
import copy
from models.ddpm import UNet
from ddpm_utils import Diffusion, train_ddpm

diffusion = Diffusion(T=1000, device=device)
ck_ema, ck_unet = "checkpoints/ddpm_ema.pt", "checkpoints/ddpm_unet.pt"
set_seed()
unet = UNet().to(device)
ema = copy.deepcopy(unet)
for p in ema.parameters():
    p.requires_grad = False
print("Jumlah parameter U-Net:", sum(p.numel() for p in unet.parameters()))   # 1012993

if os.path.exists(ck_ema):
    ema.load_state_dict(torch.load(ck_ema, map_location=device))
    print("[skip] ddpm: memuat checkpoint yang sudah ada")
else:
    print("[train] ddpm (sekitar 12 menit di T4)")
    hist = train_ddpm(unet, ema, diffusion, train_loader, device, epochs=EPOCHS)
    torch.save(unet.state_dict(), ck_unet); torch.save(ema.state_dict(), ck_ema)
    json.dump(hist, open("checkpoints/ddpm_history.json", "w"))
_ = ema.eval()

## 4. Eksperimen batasan MLP Classifier (Tahap 4 dan 5)

In [ ]:
from evaluate import evaluate_recon

m1, p1, s1 = evaluate_recon(fd, test_loader, device)
m2, p2, s2 = evaluate_recon(ae, test_loader, device)
print(f"Frozen z_MLP -> decoder | MSE {m1:.5f} | PSNR {p1:.2f} dB | SSIM {s1:.4f}")
print(f"Vanilla AE (z_AE)       | MSE {m2:.5f} | PSNR {p2:.2f} dB | SSIM {s2:.4f}")

# Statistik ruang laten MLP (Tahap 5)
with torch.no_grad():
    zs = torch.cat([clf.features(x.to(device)) for x, _ in test_loader])
mu_z, sd_z = zs.mean(0), zs.std(0)
print(f"z_MLP: min {zs.min():.2f} | max {zs.max():.2f} | fraksi tepat nol {(zs == 0).float().mean():.1%}")
torch.manual_seed(42)
z_rand = mu_z + sd_z * torch.randn(1000, 64, device=device)
print(f"z_rand: fraksi negatif {(z_rand < 0).float().mean():.1%}")

## 5. Evaluasi kuantitatif (metrik, latensi, FID, cakupan mode)

In [ ]:
import time, copy, pandas as pd
from benchmark import measure_latency

# --- Metrik rekonstruksi di test set ---
rows = {}
for name, m in [("Vanilla AE", ae), ("Sparse AE", sparse_ae), ("VAE", vae)]:
    mse, psnr, ssim = evaluate_recon(m, test_loader, device)
    rows[name] = {"MSE": mse, "PSNR": psnr, "SSIM": ssim}
    print(f"{name:10s} | MSE {mse:.5f} | PSNR {psnr:.2f} dB | SSIM {ssim:.4f}")

# --- Seberapa jarang lapisan tersembunyi 256 unit? ---
@torch.no_grad()
def zero_fraction(hidden_fn, loader):
    zeros = total = 0
    for x, _ in loader:
        h = hidden_fn(x.to(device)); zeros += (h == 0).sum().item(); total += h.numel()
    return zeros / total
print(f"Aktivasi nol | Vanilla AE: {zero_fraction(lambda x: ae.encoder[:3](x), test_loader):.1%}"
      f" | Sparse AE: {zero_fraction(lambda x: sparse_ae.encode_with_h(x)[1], test_loader):.1%}")

# --- Rata-rata waktu training per epoch (dari file history) ---
hist_files = {"Vanilla AE": "ae", "Sparse AE": "sparse_ae", "VAE": "vae",
              "Vanilla GAN": "gan", "DCGAN": "dcgan", "DDPM": "ddpm"}
epoch_time = {k: sum(json.load(open(f"checkpoints/{f}_history.json"))["epoch_time"]) /
                 len(json.load(open(f"checkpoints/{f}_history.json"))["epoch_time"]) for k, f in hist_files.items()}
print({k: round(v, 1) for k, v in epoch_time.items()})

# --- Latensi per batch 64 (GPU) ---
B = 64
x64g = next(iter(test_loader))[0][:B].to(device)
z = lambda d: torch.randn(B, d, device=device)
lat = {
    "Vanilla AE":  measure_latency(lambda: ae(x64g), device),                 # rekonstruksi (encode+decode)
    "Sparse AE":   measure_latency(lambda: sparse_ae(x64g), device),          # rekonstruksi (encode+decode)
    "VAE":         measure_latency(lambda: vae.decode(z(64)), device),        # generasi dari prior
    "Vanilla GAN": measure_latency(lambda: gan_G(z(100)), device),
    "DCGAN":       measure_latency(lambda: dc_G(z(100)), device),
    "DDPM":        measure_latency(lambda: diffusion.sample(ema, B, device), device, warmup=1, repeats=2),  # 1000 langkah
}
# --- Latensi CPU (perkiraan kasar untuk pertanyaan perangkat mobile) ---
to_cpu = lambda m: copy.deepcopy(m).cpu().eval()
vae_c, g_c, dc_c = to_cpu(vae), to_cpu(gan_G), to_cpu(dc_G)
zc = lambda d: torch.randn(B, d)
lat_cpu = {
    "VAE":         measure_latency(lambda: vae_c.decode(zc(64)), "cpu"),
    "Vanilla GAN": measure_latency(lambda: g_c(zc(100)), "cpu"),
    "DCGAN":       measure_latency(lambda: dc_c(zc(100)), "cpu"),
}
for k, v in lat.items():     print(f"Latency GPU {k:12s}: {v:10.2f} ms / batch {B}")
for k, v in lat_cpu.items(): print(f"Latency CPU {k:12s}: {v:10.2f} ms / batch {B}")
json.dump({"gpu": lat, "cpu": lat_cpu, "epoch_time": epoch_time, "recon": rows}, open("results/metrics_raw.json", "w"))

In [ ]:
# FID + sampel sintetis. Hasil disimpan; jika sudah ada di Drive, langsung dimuat (tidak dihitung ulang).
from torchmetrics.image.fid import FrechetInceptionDistance
from benchmark import generate_in_batches, compute_fid

N_FID = 5000       # HARUS sama untuk semua model
BATCH = 250

if os.path.exists("results/fid.json") and os.path.exists("results/samples_store.pt"):
    fid_scores = json.load(open("results/fid.json"))
    N_FID = fid_scores.pop("N_FID", N_FID)
    samples_store = torch.load("results/samples_store.pt")
    print(f"[skip] FID dimuat dari results/ (N_FID={N_FID})")
else:
    fid_metric = FrechetInceptionDistance(feature=2048, reset_real_features=False, normalize=True).to(device)
    with torch.no_grad():
        for x, _ in test_loader:
            fid_metric.update(x.to(device).repeat(1, 3, 1, 1), real=True)
    val_imgs = torch.cat([x for x, _ in val_loader])[:N_FID]
    fid_scores = {"Real val (baseline)": compute_fid(fid_metric, val_imgs, device)}
    print(f"Baseline real-vs-real FID = {fid_scores['Real val (baseline)']:.2f}")

    fast = {
        "VAE":         lambda b: vae.decode(torch.randn(b, 64, device=device)),
        "Vanilla GAN": lambda b: (gan_G(torch.randn(b, 100, device=device)) + 1) / 2,
        "DCGAN":       lambda b: (dc_G(torch.randn(b, 100, device=device)) + 1) / 2,
    }
    samples_store = {}
    for name, fn in fast.items():
        set_seed()
        samples_store[name] = generate_in_batches(fn, N_FID)
        fid_scores[name] = compute_fid(fid_metric, samples_store[name], device)
        print(f"{name:12s} FID = {fid_scores[name]:.2f}")

    set_seed(); imgs, t0 = [], time.time()          # DDPM: lambat, tampilkan progres
    for i in range(0, N_FID, BATCH):
        b = min(BATCH, N_FID - i)
        with torch.no_grad():
            imgs.append(((diffusion.sample(ema, b, device).clamp(-1, 1) + 1) / 2).cpu())
        print(f"DDPM {i+b}/{N_FID} | {time.time()-t0:.0f}s")
    samples_store["DDPM"] = torch.cat(imgs)
    fid_scores["DDPM"] = compute_fid(fid_metric, samples_store["DDPM"], device)
    print(f"DDPM FID = {fid_scores['DDPM']:.2f}")

    json.dump({"N_FID": N_FID, **fid_scores}, open("results/fid.json", "w"))
    torch.save(samples_store, "results/samples_store.pt")

In [ ]:
# Tabel akhir Bagian IV
names = ["Vanilla AE", "Sparse AE", "VAE", "Vanilla GAN", "DCGAN", "DDPM"]
latent = {"Vanilla AE": "64", "Sparse AE": "64 (bottleneck) / 256 (sparse)", "VAE": "64",
          "Vanilla GAN": "100 (noise)", "DCGAN": "100 (noise)", "DDPM": "28x28 (citra)"}
df = pd.DataFrame(index=names)
df["Dim Laten"]            = pd.Series(latent)
df["Epoch (s)"]            = pd.Series(epoch_time).round(1)
df["Latency GPU (ms/64)"]  = pd.Series(lat).round(2)
df["Latency CPU (ms/64)"]  = pd.Series(lat_cpu).round(2)
df["Test MSE"]             = pd.Series({k: round(v["MSE"], 5) for k, v in rows.items()})
df["Test PSNR (dB)"]       = pd.Series({k: round(v["PSNR"], 2) for k, v in rows.items()})
df["Test SSIM"]            = pd.Series({k: round(v["SSIM"], 4) for k, v in rows.items()})
df["FID"]                  = pd.Series({k: round(v, 2) for k, v in fid_scores.items() if k in names})
df = df.fillna("N/A")
print(df.to_string())
print(f"\nBaseline FID real-vs-real = {fid_scores['Real val (baseline)']:.2f} | N_FID = {N_FID}")
df.to_csv("results/metrics.csv")

In [ ]:
# Cakupan mode: klasifikasikan sampel tiap model dengan MLP Classifier, hitung distribusi kelas dan entropi ternormalisasi
import math
names10 = ["T-shirt", "Trouser", "Pullover", "Dress", "Coat", "Sandal", "Shirt", "Sneaker", "Bag", "Boot"]

@torch.no_grad()
def class_dist(imgs):
    pred = torch.cat([clf(imgs[i:i+500].to(device)).argmax(1).cpu() for i in range(0, len(imgs), 500)])
    p = torch.bincount(pred, minlength=10).float() / len(pred)
    ent = -(p[p > 0] * p[p > 0].log()).sum().item() / math.log(10)
    return p, ent

real_imgs = torch.cat([x for x, _ in test_loader])
print(" " * 12 + " ".join(f"{n[:6]:>6s}" for n in names10) + "  entropi")
for name, imgs in {"Real (test)": real_imgs, **samples_store}.items():
    p, ent = class_dist(imgs)
    print(f"{name:12s}" + " ".join(f"{v:6.0%}" for v in p.tolist()) + f"   {ent:.3f}")

## 6. Lima figur wajib (tersimpan di `figures/`)

In [ ]:
from figures import fig1_reconstruction, fig2_samples, fig3_mlp_limits, fig4_interpolation, fig5_curves_and_trajectory

x8 = next(iter(test_loader))[0][:8].to(device)
fig1_reconstruction({"Vanilla AE": ae, "Sparse AE": sparse_ae, "VAE": vae}, x8)

In [ ]:
fig2_samples({"Panel A: VAE": samples_store["VAE"], "Panel B: Vanilla GAN": samples_store["Vanilla GAN"],
              "Panel C: DCGAN": samples_store["DCGAN"], "Panel D: DDPM": samples_store["DDPM"]})

In [ ]:
fig3_mlp_limits(clf, fd, ae, test_loader, device)

In [ ]:
fig4_interpolation(ae, sparse_ae, vae, gan_G, dc_G, clf, test_loader, device)   # kelas 1 (Trouser) vs 7 (Sneaker)

In [ ]:
fig5_curves_and_trajectory(diffusion, ema, device)

## 7. Paket untuk GitHub dan cadangan

In [ ]:
!zip -q -r code_for_github.zip data.py evaluate.py benchmark.py train_utils.py gan_utils.py ddpm_utils.py figures.py models requirements.txt README.md
!zip -q -r results_figures.zip results/metrics.csv results/fid.json results/metrics_raw.json figures
from google.colab import files
files.download("code_for_github.zip")
files.download("results_figures.zip")